# Day 1 — MIMIC-IV Data Validation

Objective:
Validate the local MIMIC-IV dataset and understand the identifiers and relationships that will form the foundation of TrialMatch AI.

TrialMatch AI will eventually map clinical trial eligibility criteria to patient evidence and classify criteria as:

- **SATISFIED**
- **NOT_SATISFIED**
- **UNKNOWN**

Today we are only validating the data foundation.

## Section 2 — Setup

In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"
if not DATA_DIR.exists():
    DATA_DIR = Path("data")  # fallback if the kernel runs from the project root

print("Resolved DATA_DIR:", DATA_DIR.resolve())

Resolved DATA_DIR: /Users/spandanarangaswamy/PROJECTS-/TrialMatch_AI/data


## Section 3 — Verify expected files

In [2]:
expected_files = [
    "patients.csv.gz",
    "admissions.csv.gz",
    "diagnoses_icd.csv.gz",
    "d_icd_diagnoses.csv.gz",
    "d_labitems.csv.gz",
    "labevents.csv.gz",
    "procedures_icd.csv",
    "procedures_icd.csv.gz",
    "d_icd_procedures.csv.gz",
    "prescriptions.csv.gz",
    "emar.csv.gz",
    "emar_detail.csv.gz",
    "omr.csv.gz",
    "icustays.csv.gz",
    "chartevents.csv.gz",
    "d_items.csv.gz",
]

rows = []
for name in expected_files:
    path = DATA_DIR / name
    if path.exists():
        rows.append({"file": name, "exists": True, "size_mb": round(path.stat().st_size / 1e6, 2)})
    else:
        rows.append({"file": name, "exists": False, "size_mb": None})

pd.DataFrame(rows)

# procedures_icd is valid if either the plain csv or the gzipped version exists
has_procedures = (DATA_DIR / "procedures_icd.csv").exists() or (DATA_DIR / "procedures_icd.csv.gz").exists()
print("procedures_icd present (csv or csv.gz):", has_procedures)

procedures_icd present (csv or csv.gz): True


## Section 4 — Load core tables

Only the four small core tables are loaded here. Large event-level files
(`labevents`, `chartevents`, `emar`, `emar_detail`) are **not** loaded — they are
unnecessary for Day 1 validation and expensive to read.

In [3]:
patients = pd.read_csv(DATA_DIR / "patients.csv.gz")
admissions = pd.read_csv(DATA_DIR / "admissions.csv.gz")
diagnoses = pd.read_csv(DATA_DIR / "diagnoses_icd.csv.gz")
diagnosis_dictionary = pd.read_csv(DATA_DIR / "d_icd_diagnoses.csv.gz")

print("patients:", patients.shape)
print("admissions:", admissions.shape)
print("diagnoses:", diagnoses.shape)
print("diagnosis_dictionary:", diagnosis_dictionary.shape)

patients: (364627, 6)
admissions: (546028, 16)
diagnoses: (6364488, 5)
diagnosis_dictionary: (112107, 3)


## Section 5 — Inspect patients

`subject_id` is the unique patient identifier: every row in `patients` is one
person, and every piece of patient-level information elsewhere in MIMIC-IV
links back to it.

In [4]:
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [5]:
print("shape:", patients.shape)
print("columns:", patients.columns.tolist())
print(patients.dtypes)

print("rows:", len(patients))
print("unique subject_id:", patients["subject_id"].nunique())
print("subject_id is unique:", patients["subject_id"].is_unique)

shape: (364627, 6)
columns: ['subject_id', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod']
subject_id            int64
gender               object
anchor_age            int64
anchor_year           int64
anchor_year_group    object
dod                  object
dtype: object
rows: 364627
unique subject_id: 364627
subject_id is unique: True


## Section 6 — Inspect admissions

- `subject_id` identifies the patient.
- `hadm_id` identifies one hospitalization.
- One patient may have multiple hospitalizations, so `admissions` has one row per stay, not per person.

In [6]:
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


In [7]:
print("shape:", admissions.shape)
print("columns:", admissions.columns.tolist())

print("unique subject_id:", admissions["subject_id"].nunique())
print("unique hadm_id:", admissions["hadm_id"].nunique())
print("hadm_id is unique:", admissions["hadm_id"].is_unique)

shape: (546028, 16)
columns: ['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admit_provider_id', 'admission_location', 'discharge_location', 'insurance', 'language', 'marital_status', 'race', 'edregtime', 'edouttime', 'hospital_expire_flag']
unique subject_id: 223452
unique hadm_id: 546028
hadm_id is unique: True


## Section 7 — Validate patient → admission relationship

Every hospitalization must belong to a known patient. We check that below,
then look at a few examples of patients with multiple admissions (a small
sample only — no population-level statistics).

In [8]:
missing_subjects = set(admissions["subject_id"]) - set(patients["subject_id"])
print("admission subject_ids not found in patients:", len(missing_subjects))

admissions_per_patient = admissions.groupby("subject_id").size().sort_values(ascending=False)
print("\nExample: patients with the most admissions (sample only):")
admissions_per_patient.head(10)

admission subject_ids not found in patients: 0

Example: patients with the most admissions (sample only):


subject_id
15496609    238
15464144    185
10714009    163
16662316    142
14394983    138
15229574    130
11582633    105
17011846    104
13475033    103
16233333    101
dtype: int64

## Section 8 — Inspect diagnosis structure

`diagnoses_icd` contains coded diagnoses associated with hospitalizations.
`d_icd_diagnoses` translates ICD codes into human-readable descriptions.

In [9]:
diagnoses.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [10]:
diagnosis_dictionary.head()

,icd_code,icd_version,long_title
0,0010,9,Cholera due to vibrio cholerae
1,0011,9,Cholera due to vibrio cholerae el tor
2,0019,9,"Cholera, unspecified"
3,0020,9,Typhoid fever
4,0021,9,Paratyphoid fever A


In [11]:
print("diagnoses columns:", diagnoses.columns.tolist())
print("diagnosis_dictionary columns:", diagnosis_dictionary.columns.tolist())

diagnoses columns: ['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version']
diagnosis_dictionary columns: ['icd_code', 'icd_version', 'long_title']


## Section 9 — Join diagnoses to descriptions

The dictionary is joined on **both** `icd_code` and `icd_version`, because the
same code text means different things in ICD-9 vs ICD-10. No specific disease
is searched for yet.

In [12]:
diagnoses_labeled = diagnoses.merge(
    diagnosis_dictionary,
    on=["icd_code", "icd_version"],
    how="left",
)

diagnoses_labeled[[
    "subject_id",
    "hadm_id",
    "seq_num",
    "icd_code",
    "icd_version",
    "long_title",
]].head(10)

,subject_id,hadm_id,seq_num,icd_code,icd_version,long_title
0,10000032,22595853,1,5723,9,Portal hypertension
1,10000032,22595853,2,78959,9,Other ascites
2,10000032,22595853,3,5715,9,Cirrhosis of liver without mention of alcohol
3,10000032,22595853,4,07070,9,Unspecified viral hepatitis C without hepatic ...
4,10000032,22595853,5,496,9,"Chronic airway obstruction, not elsewhere clas..."
5,10000032,22595853,6,29680,9,"Bipolar disorder, unspecified"
6,10000032,22595853,7,30981,9,Posttraumatic stress disorder
7,10000032,22595853,8,V1582,9,Personal history of tobacco use
8,10000032,22841357,1,07071,9,Unspecified viral hepatitis C with hepatic coma
9,10000032,22841357,2,78959,9,Other ascites


## Section 10 — Validate diagnosis relationships

Referential checks: diagnoses must point at real patients, at real
hospitalizations, and their codes should be resolvable through the dictionary.
Unmatched records are counted, never silently dropped.

In [13]:
missing_dx_subjects = set(diagnoses["subject_id"]) - set(patients["subject_id"])
print("1. diagnosis subject_ids not found in patients:", len(missing_dx_subjects))

non_null_hadm = diagnoses.loc[diagnoses["hadm_id"].notna(), "hadm_id"]
missing_hadm = set(non_null_hadm) - set(admissions["hadm_id"])
print("2. non-null diagnosis hadm_ids not found in admissions:", len(missing_hadm))

unlabeled = int(diagnoses_labeled["long_title"].isna().sum())
print("3. diagnosis rows with no long_title after dictionary join:", unlabeled)

1. diagnosis subject_ids not found in patients: 0


2. non-null diagnosis hadm_ids not found in admissions: 0
3. diagnosis rows with no long_title after dictionary join: 0


## Section 11 — One patient walkthrough

This is the basic evidence path TrialMatch AI will eventually use:
a patient row identifies the person, their admissions locate the
hospitalizations, and the diagnoses attached to each hospitalization provide
coded clinical evidence. Eligibility criteria will be evaluated against
exactly this chain.

In [14]:
walkthrough_subject = int(diagnoses["subject_id"].iloc[0])
print("Patient subject_id:", walkthrough_subject)

patients.loc[patients["subject_id"] == walkthrough_subject]

Patient subject_id: 10000032


,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09


In [15]:
admissions.loc[
    admissions["subject_id"] == walkthrough_subject,
    ["subject_id", "hadm_id", "admittime", "dischtime", "admission_type"],
]

,subject_id,hadm_id,admittime,dischtime,admission_type
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,URGENT
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,EW EMER.
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,EW EMER.


In [16]:
diagnoses_labeled.loc[
    diagnoses_labeled["subject_id"] == walkthrough_subject,
    ["hadm_id", "seq_num", "icd_code", "icd_version", "long_title"],
].head(10)

,hadm_id,seq_num,icd_code,icd_version,long_title
0,22595853,1,5723,9,Portal hypertension
1,22595853,2,78959,9,Other ascites
2,22595853,3,5715,9,Cirrhosis of liver without mention of alcohol
3,22595853,4,07070,9,Unspecified viral hepatitis C without hepatic ...
4,22595853,5,496,9,"Chronic airway obstruction, not elsewhere clas..."
5,22595853,6,29680,9,"Bipolar disorder, unspecified"
6,22595853,7,30981,9,Posttraumatic stress disorder
7,22595853,8,V1582,9,Personal history of tobacco use
8,22841357,1,07071,9,Unspecified viral hepatitis C with hepatic coma
9,22841357,2,78959,9,Other ascites


# Day 1 Takeaways

- MIMIC-IV is relational rather than one large patient table.
- `subject_id` identifies a patient.
- `hadm_id` identifies a hospitalization.
- One patient may have multiple hospitalizations.
- Clinical evidence such as diagnoses is stored separately from demographics.
- Dictionary tables translate coded concepts into readable clinical concepts.
- Future TrialMatch logic will retrieve evidence across multiple tables.
- Missing evidence must eventually be distinguished from negative evidence.

The identifier chain that TrialMatch AI will follow:

```
Patient
  │
  └── subject_id
        │
        ├── Admission
        │     └── hadm_id
        │           └── Diagnoses
        │
        └── Admission
              └── hadm_id
                    └── Diagnoses
```